In [1]:
!rm -rf /kagg/working/*

In [2]:
import warnings
import os
import sys
import logging

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

class SuppressDataLoaderWarnings:
    def __init__(self, stream):
        self.stream = stream
        self._suppress_keywords = [
            "can only test a child process",
            "AssertionError",
            "Exception ignored in",
            "_MultiProcessingDataLoaderIter",
            "_shutdown_workers",
        ]

    def write(self, msg):
        if not any(keyword in msg for keyword in self._suppress_keywords):
            self.stream.write(msg)

    def flush(self):
        self.stream.flush()

    def fileno(self):
        return self.stream.fileno()

sys.stderr = SuppressDataLoaderWarnings(sys.stderr)
os.environ["PYTHONWARNINGS"] = "ignore"


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
: 

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
: 

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
: 

Traceback (most recent call last):
  File "/usr/local/

In [3]:
import os
import glob
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.model_selection import GroupKFold
import warnings
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
from torchinfo import summary

from tqdm.notebook import tqdm
from tabulate import tabulate
from IPython.display import clear_output
from tqdm.notebook import tqdm
from tabulate import tabulate
import os, re, glob, warnings
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import timm
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.isotonic import IsotonicRegression
from torchinfo import summary
from tabulate import tabulate
from tqdm.auto import tqdm
from IPython.display import clear_output
warnings.filterwarnings('ignore')
from IPython.display import clear_output
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")

In [4]:
CONFIG = {
    'seed': 42,
    'img_size': 384,
    'batch_size': 2,
    'epochs': 30,
    'lr': 0.005,
    'min_lr': 1e-7,
    'weight_decay': 1e-4,
    'n_splits': 5,
    'num_workers': 2,
    'tta_n': 5,
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'data_dir': '/kaggle/input/competitions/soil-grain-size-from-photos',
    'work_dir': '/kaggle/working',
    'model_name': 'swinv2_large_window12to24_192to384.ms_in22k_ft_in1k',
    'label_cols': ['0.002','0.0063','0.02','0.063','0.2','0.63','2','6.3','20','63','200'],
}

In [5]:
def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CONFIG['seed'])

GRAIN_DIAMETERS = torch.tensor([0.002,0.0063,0.02,0.063,0.2,0.63,2.0,6.3,20.0,63.0,200.0])
LOG_DIFFS = (torch.log10(GRAIN_DIAMETERS[1:]) - torch.log10(GRAIN_DIAMETERS[:-1])).to(CONFIG['device'])

In [6]:
train_label = pd.read_csv(os.path.join(CONFIG['data_dir'], 'Training_labels_without_H374.csv'))
ppm_csv = pd.read_csv(os.path.join(CONFIG['data_dir'], 'ppm.csv'))
sub_template = pd.read_csv(os.path.join(CONFIG['data_dir'], 'sample_submission.csv'))

ppm_dict = {}
for _, row in ppm_csv.iterrows():
    ppm_dict[str(row['camera'])] = float(row['ppm'])
    ppm_dict[str(row['phone'])] = float(row['ppm'])

MEAN_PPM = float(np.mean(list(ppm_dict.values())))

print(f"Train samples: {len(train_label)}")
print(f"Test samples: {len(sub_template)}")
print(f"PPM cameras: {len(ppm_dict)}")
train_label.head()

Train samples: 25
Test samples: 10
PPM cameras: 6


,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,F827,9.4904,19.3892,47.6257,89.5554,99.8965,99.9896,100.0000,100.0000,100.0000,100.0000,100.0
1,G190,5.2076,10.2425,23.7537,50.3113,75.8311,85.7233,90.9898,94.8969,98.5814,100.0000,100.0
2,H030,6.7833,10.9091,19.0599,37.3235,66.8349,96.4965,97.9530,99.5691,100.0000,100.0000,100.0
3,H031,1.8556,5.5638,11.4945,19.1022,25.8591,39.6389,50.5450,65.2738,81.4215,94.7783,100.0
4,H037,0.9320,6.4500,16.3216,27.8019,34.6629,46.3652,62.7168,79.0523,89.7830,100.0000,100.0


In [7]:
train_transforms = T.Compose([
    T.RandomResizedCrop(CONFIG['img_size'], scale=(0.5, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(90),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

valid_transforms = T.Compose([
    T.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

tta_transforms = T.Compose([
    T.RandomResizedCrop(CONFIG['img_size'], scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [8]:
class SoilDataset(Dataset):
    def __init__(self, image_paths, ppm_dict, mean_ppm, labels_df=None, transform=None):
        self.image_paths = image_paths
        self.ppm_dict = ppm_dict
        self.mean_ppm = mean_ppm
        self.labels_df = labels_df
        self.transform = transform
        
        if labels_df is not None:
            df = labels_df.copy()
            if 'sample_id' in df.columns:
                df = df.set_index('sample_id')
            df.index = df.index.astype(str).str.upper()
            self.labels_df = df

    def _get_ppm(self, filename):
        for cam in self.ppm_dict:
            if cam in filename:
                return self.ppm_dict[cam]
        return self.mean_ppm

    def _get_sample_id(self, filename):
        for sid in self.labels_df.index:
            if sid in filename.upper():
                return sid
        return None

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        
        filename = os.path.basename(img_path).upper()
        ppm_val = self._get_ppm(os.path.basename(img_path))
        ppm_tensor = torch.tensor([ppm_val / MEAN_PPM], dtype=torch.float32)
        
        if self.labels_df is not None:
            sid = self._get_sample_id(filename)
            targets = self.labels_df.loc[sid, CONFIG['label_cols']].values.astype(np.float32)
            return image, ppm_tensor, torch.tensor(targets), sid
        else:
            match = re.search(r'TEST_\d+|[A-Z]\d{3}', filename)
            sid = match.group(0) if match else filename.split('_')[0]
            return image, ppm_tensor, sid

In [9]:
class SoilGrainModel(nn.Module):
    def __init__(self, model_name=CONFIG['model_name']):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        in_features = self.backbone.num_features
        
        self.ppm_fc = nn.Sequential(
            nn.Linear(1, 32),
            nn.GELU(),
            nn.Linear(32, 64),
            nn.GELU(),
        )
        
        self.head = nn.Sequential(
            nn.Linear(in_features + 64, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 11)
        )

    def forward(self, x_img, x_ppm):
        img_feats = self.backbone(x_img)
        ppm_feats = self.ppm_fc(x_ppm)
        x = torch.cat([img_feats, ppm_feats], dim=1)
        logits = self.head(x)
        fracs = torch.softmax(logits, dim=1)
        cum = torch.cumsum(fracs, dim=1)
        return cum * 100.0

model = SoilGrainModel().to(CONFIG['device'])
summary(model, input_data=(torch.randn(2,3,CONFIG['img_size'],CONFIG['img_size']).to(CONFIG['device']),torch.randn(2,1).to(CONFIG['device'])), depth=3, verbose=1)

Layer (type:depth-idx)                                  Output Shape              Param #
SoilGrainModel                                          [2, 11]                   --
├─SwinTransformerV2: 1-1                                [2, 1536]                 --
│    └─PatchEmbed: 2-1                                  [2, 96, 96, 192]          --
│    │    └─Conv2d: 3-1                                 [2, 192, 96, 96]          9,408
│    │    └─LayerNorm: 3-2                              [2, 96, 96, 192]          384
│    └─Sequential: 2-2                                  [2, 12, 12, 1536]         --
│    │    └─SwinTransformerV2Stage: 3-3                 [2, 96, 96, 192]          898,572
│    │    └─SwinTransformerV2Stage: 3-4                 [2, 48, 48, 384]          3,859,224
│    │    └─SwinTransformerV2Stage: 3-5                 [2, 24, 24, 768]          128,998,320
│    │    └─SwinTransformerV2Stage: 3-6                 [2, 12, 12, 1536]         61,433,952
│    └─LayerNorm: 2-3      

Layer (type:depth-idx)                                  Output Shape              Param #
SoilGrainModel                                          [2, 11]                   --
├─SwinTransformerV2: 1-1                                [2, 1536]                 --
│    └─PatchEmbed: 2-1                                  [2, 96, 96, 192]          --
│    │    └─Conv2d: 3-1                                 [2, 192, 96, 96]          9,408
│    │    └─LayerNorm: 3-2                              [2, 96, 96, 192]          384
│    └─Sequential: 2-2                                  [2, 12, 12, 1536]         --
│    │    └─SwinTransformerV2Stage: 3-3                 [2, 96, 96, 192]          898,572
│    │    └─SwinTransformerV2Stage: 3-4                 [2, 48, 48, 384]          3,859,224
│    │    └─SwinTransformerV2Stage: 3-5                 [2, 24, 24, 768]          128,998,320
│    │    └─SwinTransformerV2Stage: 3-6                 [2, 12, 12, 1536]         61,433,952
│    └─LayerNorm: 2-3      

In [10]:
class LogWeightedEMDLoss(nn.Module):
    def __init__(self, smoothing=0.0):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, preds, targets):
        if self.smoothing > 0:
            targets = targets * (1 - self.smoothing) + self.smoothing * 50.0
        abs_diff = torch.abs(preds[:, :-1] - targets[:, :-1])
        emd = torch.sum(abs_diff * LOG_DIFFS, dim=1)
        return emd.mean()

class EarlyStopping:
    def __init__(self, patience=6, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [11]:
all_train_images = np.array(sorted(glob.glob(
    os.path.join(CONFIG['data_dir'], 'Training-All_Photos_without_H374/**/*.jpg'), recursive=True
)))

train_groups = []
for path in all_train_images:
    filename = os.path.basename(path).upper()
    sid = next((s for s in train_label['sample_id'].astype(str).str.upper() if s in filename), None)
    train_groups.append(sid)
train_groups = np.array(train_groups)

print(f"Total training images: {len(all_train_images)}")
print(f"Unique sample groups: {len(set(g for g in train_groups if g))}")

Total training images: 130
Unique sample groups: 24


In [12]:
gkf = GroupKFold(n_splits=CONFIG['n_splits'])
oof_preds = {}
fold_val_losses = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(all_train_images, groups=train_groups)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{CONFIG['n_splits']}")
    print(f"{'='*60}")
    
    train_paths = all_train_images[train_idx]
    val_paths   = all_train_images[val_idx]
    
    train_ds = SoilDataset(train_paths, ppm_dict, MEAN_PPM, train_label, train_transforms)
    val_ds   = SoilDataset(val_paths,   ppm_dict, MEAN_PPM, train_label, valid_transforms)
    
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=CONFIG['num_workers'], pin_memory=True, persistent_workers=True)
    
    model = SoilGrainModel().to(CONFIG['device'])
    criterion = LogWeightedEMDLoss(smoothing=0.0)
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=CONFIG['min_lr'])
    scaler = GradScaler()
    early_stopping = EarlyStopping(patience=6)
    
    best_val_loss = float('inf')
    history = []
    best_model_path = os.path.join(CONFIG['work_dir'], f'best_fold{fold+1}.pth')
    
    for epoch in range(CONFIG['epochs']):
        model.train()
        train_loss = 0.0
        
        for images, ppms, targets, _ in tqdm(train_loader, desc=f"E{epoch+1} Train", leave=False):
            images  = images.to(CONFIG['device'])
            ppms    = ppms.to(CONFIG['device'])
            targets = targets.to(CONFIG['device'])
            
            optimizer.zero_grad()
            with autocast():
                preds = model(images, ppms)
                loss  = criterion(preds, targets)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        
        scheduler.step()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, ppms, targets, _ in tqdm(val_loader, desc=f"E{epoch+1} Val", leave=False):
                images  = images.to(CONFIG['device'])
                ppms    = ppms.to(CONFIG['device'])
                targets = targets.to(CONFIG['device'])
                preds   = model(images, ppms)
                val_loss += criterion(preds, targets).item()
        
        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        
        is_best = ""
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            is_best = "★"
        
        history.append([epoch+1, f"{avg_train:.4f}", f"{avg_val:.4f}",
                        f"{scheduler.get_last_lr()[0]:.2e}", is_best])
        
        clear_output(wait=True)
        print(f"FOLD {fold+1} | Best Val: {best_val_loss:.4f}")
        print(tabulate(history[-10:],
                       headers=["Epoch","Train EMD","Val EMD","LR","Best"],
                       tablefmt="heavy_grid", stralign="center", numalign="center"))
        
        early_stopping(avg_val)
        if early_stopping.early_stop:
            print(f"Early stop at epoch {epoch+1}")
            break
    
    fold_val_losses.append(best_val_loss)
    
    model.load_state_dict(torch.load(best_model_path, map_location=CONFIG['device']))
    model.eval()
    with torch.no_grad():
        for images, ppms, targets, sids in val_loader:
            images = images.to(CONFIG['device'])
            ppms   = ppms.to(CONFIG['device'])
            preds  = model(images, ppms).cpu().numpy()
            for sid, pred in zip(sids, preds):
                if sid not in oof_preds:
                    oof_preds[sid] = []
                oof_preds[sid].append(pred)

print(f"\nFold Val Losses: {[f'{l:.4f}' for l in fold_val_losses]}")
print(f"Mean CV EMD: {np.mean(fold_val_losses):.4f} ± {np.std(fold_val_losses):.4f}")

FOLD 5 | Best Val: 69.6805
┏━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓
┃  Epoch  ┃  Train EMD  ┃  Val EMD  ┃   LR    ┃  Best  ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━┫
┃    1    ┃   102.122   ┃  69.6805  ┃ 0.00488 ┃   ★    ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━┫
┃    2    ┃   98.9038   ┃  101.456  ┃ 0.00452 ┃        ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━┫
┃    3    ┃   97.7423   ┃  96.8933  ┃ 0.00397 ┃        ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━┫
┃    4    ┃   95.9622   ┃  103.082  ┃ 0.00327 ┃        ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━┫
┃    5    ┃   95.0596   ┃  112.14   ┃ 0.0025  ┃        ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━┫
┃    6    ┃   85.9714   ┃  100.239  ┃ 0.00173 ┃        ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━┫
┃    7    ┃   89.0548   ┃  96.9534  ┃ 0.00103 ┃        ┃
┗━━━━━━━━━┻━━━━━━━━━━━━━┻━━━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━┛
Earl

In [13]:
def enforce_monotone(arr):
    ir = IsotonicRegression(increasing=True, out_of_bounds='clip')
    return ir.fit_transform(np.arange(len(arr)), arr)

label_df_indexed = train_label.set_index('sample_id')
label_df_indexed.index = label_df_indexed.index.astype(str).str.upper()

oof_emd_list = []
for sid, preds_list in oof_preds.items():
    avg_pred = np.mean(preds_list, axis=0)
    avg_pred = enforce_monotone(avg_pred)
    avg_pred[-1] = 100.0
    if sid in label_df_indexed.index:
        true_vals = label_df_indexed.loc[sid, CONFIG['label_cols']].values.astype(np.float32)
        diff = np.abs(avg_pred[:-1] - true_vals[:-1])
        emd = np.sum(diff * LOG_DIFFS.cpu().numpy())
        oof_emd_list.append(emd)

print(f"OOF EMD: {np.mean(oof_emd_list):.4f} ± {np.std(oof_emd_list):.4f}")
print(f"OOF samples evaluated: {len(oof_emd_list)}")

OOF EMD: 82.2527 ± 40.4853
OOF samples evaluated: 24


In [14]:
import unicodedata

def normalize_str(s):
    umlaut_map = {'ü':'ue','ö':'oe','ä':'ae','Ü':'UE','Ö':'OE','Ä':'AE','ß':'ss'}
    for k, v in umlaut_map.items():
        s = s.replace(k, v)
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = s.replace(',','_').replace('.','_').replace(' ','_').replace('(','_').replace(')','_')
    return s.upper()

test_dir = os.path.join(CONFIG['data_dir'], 'Test_All_Photos/Test_All_Photos')
test_images_paths = sorted([
    os.path.join(test_dir, f)
    for f in os.listdir(test_dir)
    if f.upper().endswith('.JPG')
])
print(f"Test images found: {len(test_images_paths)}")

valid_test_ids_original = sub_template['sample_id'].astype(str).values
norm_to_original = {normalize_str(s): s for s in valid_test_ids_original}

class TestDataset(Dataset):
    def __init__(self, image_paths, ppm_dict, mean_ppm, transform):
        self.image_paths = image_paths
        self.ppm_dict    = ppm_dict
        self.mean_ppm    = mean_ppm
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image    = Image.open(img_path).convert("RGB")
        image    = self.transform(image)
        filename = os.path.basename(img_path)
        ppm_val  = next((self.ppm_dict[c] for c in self.ppm_dict if c in filename), self.mean_ppm)
        ppm_t    = torch.tensor([ppm_val / self.mean_ppm], dtype=torch.float32)
        fn_norm  = normalize_str(filename)
        sid      = next((orig for norm_id, orig in norm_to_original.items() if norm_id in fn_norm), "UNKNOWN")
        return image, ppm_t, sid

all_test_preds = {}

for fold in range(CONFIG['n_splits']):
    best_model_path = os.path.join(CONFIG['work_dir'], f'best_fold{fold+1}.pth')
    if not os.path.exists(best_model_path):
        print(f"Fold {fold+1} model not found, skipping.")
        continue
    fold_model = SoilGrainModel().to(CONFIG['device'])
    fold_model.load_state_dict(torch.load(best_model_path, map_location=CONFIG['device']))
    fold_model.eval()
    print(f"Loaded fold {fold+1}")

    for tta_i in range(CONFIG['tta_n']):
        t  = tta_transforms if tta_i > 0 else valid_transforms
        ds = TestDataset(test_images_paths, ppm_dict, MEAN_PPM, t)
        dl = DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=False,
                        num_workers=2, pin_memory=True, persistent_workers=True)
        with torch.no_grad():
            for images, ppms, sids in tqdm(dl, desc=f"Fold{fold+1} TTA{tta_i}", leave=False):
                images = images.to(CONFIG['device'])
                ppms   = ppms.to(CONFIG['device'])
                preds  = fold_model(images, ppms).cpu().numpy()
                for sid, pred in zip(sids, preds):
                    if sid == "UNKNOWN":
                        continue
                    if sid not in all_test_preds:
                        all_test_preds[sid] = []
                    all_test_preds[sid].append(pred)

print(f"Test samples with predictions: {len(all_test_preds)}")
print(f"Sample IDs found: {list(all_test_preds.keys())}")

Test images found: 35
Loaded fold 1


Fold1 TTA0:   0%|          | 0/18 [00:00<?, ?it/s]

Fold1 TTA1:   0%|          | 0/18 [00:00<?, ?it/s]

Fold1 TTA2:   0%|          | 0/18 [00:00<?, ?it/s]

Fold1 TTA3:   0%|          | 0/18 [00:00<?, ?it/s]

...

Fold5 TTA1:   0%|          | 0/18 [00:00<?, ?it/s]

Fold5 TTA2:   0%|          | 0/18 [00:00<?, ?it/s]

Fold5 TTA3:   0%|          | 0/18 [00:00<?, ?it/s]

Fold5 TTA4:   0%|          | 0/18 [00:10<?, ?it/s]

Test samples with predictions: 10
Sample IDs found: ['HPC_Audorfring', 'HPC_Muenster_BS6_9_0-10m', 'HPC_Testfeld Lidl WHV', 'HPC_Airbus BS10-4bis7', 'HPC_Airbus BS12-5bis8', 'HPC_Airbus BS6-3', 'HPC_Kleinkummerfeld 18-3', 'HPC_Kleinkummerfeld 2-2', 'HPC_Kleinkummerfeld 2-3', 'HPC_Kleinkummerfeld 9-4']


In [15]:
pred_cols = list(sub_template.columns[1:])
rows = []

for orig_sid in valid_test_ids_original:
    if orig_sid in all_test_preds:
        avg = np.mean(all_test_preds[orig_sid], axis=0)
    else:
        print(f"WARNING: No prediction for {orig_sid}, using baseline")
        avg = np.array([9.09,18.18,27.27,36.36,45.45,54.55,63.64,72.73,81.82,90.91,100.0])

    avg = enforce_monotone(avg)
    avg = np.clip(avg, 0, 100)
    avg[-1] = 100.0
    rows.append([orig_sid] + avg.tolist())

final_sub = pd.DataFrame(rows, columns=['sample_id'] + pred_cols)

assert list(final_sub.columns) == list(sub_template.columns), "Column mismatch!"
assert (final_sub.iloc[:, 1:].diff(axis=1).iloc[:, 1:] >= 0).all().all(), "Monotonicity violated!"
assert (final_sub.iloc[:, -1] == 100.0).all(), "Last column not 100!"
assert final_sub['sample_id'].tolist() == sub_template['sample_id'].tolist(), "Sample ID mismatch!"

out_path = os.path.join(CONFIG['work_dir'], 'submission.csv')
final_sub.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {final_sub.shape}")
final_sub.head()

Saved: /kaggle/working/submission.csv
Shape: (10, 12)


,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,0.842866,1.645244,4.320408,7.557292,39.700272,54.595577,78.094040,84.536644,96.729973,99.467354,100.0
1,HPC_Audorfring,0.842866,1.645243,4.320409,7.557290,39.700165,54.595390,78.093849,84.536522,96.729973,99.467323,100.0
2,HPC_Muenster_BS6_9_0-10m,0.842869,1.645255,4.320421,7.557323,39.699726,54.594704,78.093399,84.535988,96.729996,99.467316,100.0
3,HPC_Airbus BS10-4bis7,0.842866,1.645245,4.320410,7.557297,39.700329,54.595890,78.094269,84.536957,96.729973,99.467354,100.0
4,HPC_Airbus BS12-5bis8,0.842866,1.645244,4.320406,7.557297,39.700222,54.595810,78.094238,84.536957,96.729973,99.467354,100.0
